# NRSur7dq4 / IMRPhenomXPHM fitting factor with BMS-frame preprocessing

This notebook computes a fixed-candidate fitting factor between an NRSur7dq4 target and an IMRPhenomXPHM candidate twice: first directly, then after running the BMS-frame preprocessing hook. The opt-in synchronization path defaults to aligning the candidate radiated-angular-momentum direction to the target with a coherent Wigner rotation in `ModesArray` mode space. Linear-momentum and angular-then-linear synchronization remain explicit alternatives. Boosts, supertranslations, and charge equalization remain future options.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import waveformtools.comparison  # installs ModesArray comparison helpers
from waveformtools.bms_frame_preprocessing import (
    bms_synchronized_fixed_candidate_fitting_factor,
    preprocess_bms_frame,
)
from waveformtools.comparison import (
    AlignmentSpec,
    ModeComparisonConfig,
    RotationSpec,
    fixed_candidate_fitting_factor,
    prepare_aligned_mode_data,
)
from waveformtools.models.lal import LALWaveformModel


## Waveform parameters

In [ ]:
COMMON_PARAMETERS = {
    "mass1": 40.0,
    "mass2": 35.0,
    "spin1x": 0.08,
    "spin1y": -0.03,
    "spin1z": 0.25,
    "spin2x": -0.04,
    "spin2y": 0.02,
    "spin2z": -0.15,
    "distance": 400.0,
    "inclination": 0.7,
    "phi_ref": 0.2,
    "f_ref": 20.0,
    "f_max": 512.0,
    "delta_t": 1.0 / 2048.0,
    "delta_f": 1.0 / 4.0,
    "ell_max": 2,
}

NRSUR_PARAMETERS = {
    **COMMON_PARAMETERS,
    "approximant": "NRSur7dq4",
    "f_lower": 0.0,
}
PHENOM_PARAMETERS = {
    **COMMON_PARAMETERS,
    "approximant": "IMRPhenomXPHM",
    "f_lower": 20.0,
}

BMS_DIAGNOSTICS = {
    "initial_mass": 1.0,
    "final_mass": 0.95,
    "compute_memory_finite_time": False,
}


## Helpers

In [ ]:
def generate_lal_modes(parameters):
    model = LALWaveformModel(parameters_dict=dict(parameters))
    return model.get_td_waveform_modes(dimensionless=True)


def available_modes(modes_obj, ell_min=2, ell_max=None):
    if ell_max is None:
        ell_max = int(getattr(modes_obj, "ell_max", ell_min))
    modes = []
    for ell in range(int(ell_min), int(ell_max) + 1):
        for emm in range(-ell, ell + 1):
            try:
                data = np.asarray(modes_obj.mode(ell, emm))
            except Exception:
                continue
            if data.size and np.any(np.isfinite(data)):
                modes.append((ell, emm))
    return modes


def common_available_modes(target_modes, candidate_modes, ell_min=2, ell_max=2):
    target = set(available_modes(target_modes, ell_min=ell_min, ell_max=ell_max))
    candidate = set(available_modes(candidate_modes, ell_min=ell_min, ell_max=ell_max))
    modes = sorted(target.intersection(candidate))
    if not modes:
        raise RuntimeError("No common finite modes found.")
    return modes


def normalized_rms_residue(target_modes, candidate_modes, selected_modes, result):
    aligned = prepare_aligned_mode_data(
        target_modes,
        candidate_modes,
        result,
        modes=selected_modes,
    )
    numerator = 0.0
    denominator = 0.0
    for mode in aligned.selected_modes:
        diff = aligned.reference_modes[mode] - aligned.candidate_modes[mode]
        numerator += float(np.mean(np.abs(diff) ** 2))
        denominator += float(np.mean(np.abs(aligned.reference_modes[mode]) ** 2))
    return float(np.sqrt(numerator / max(denominator, 1e-300)))


def print_fitting_result(label, result, rms_residue):
    print(label)
    print("  match:", result.match)
    print("  mismatch:", result.mismatch)
    print("  normalized RMS residue:", rms_residue)
    print("  best alignment parameters:", result.best_parameters["alignment"])


## Generate the two waveform modes objects

In [ ]:
target_modes = generate_lal_modes(NRSUR_PARAMETERS)
candidate_modes = generate_lal_modes(PHENOM_PARAMETERS)
selected_modes = common_available_modes(target_modes, candidate_modes, ell_max=2)

print("Selected modes:", selected_modes)
print("NRSur samples:", len(target_modes.time_axis))
print("IMRPhenomXPHM samples:", len(candidate_modes.time_axis))
print("NRSur time range:", (target_modes.time_axis[0], target_modes.time_axis[-1]))
print("IMRPhenomXPHM time range:", (candidate_modes.time_axis[0], candidate_modes.time_axis[-1]))
print("NRSur max |h22|:", np.max(np.abs(target_modes.mode(2, 2))))
print("IMRPhenomXPHM max |h22|:", np.max(np.abs(candidate_modes.mode(2, 2))))


## Fitting factor without BMS-frame preprocessing

In [ ]:
comparison = ModeComparisonConfig(
    modes=selected_modes,
    alignment=AlignmentSpec(
        time_alignment="peak_total_news_power",
        time_domain_policy="resample_to_reference",
        phase_alignment="orbital_phase_and_global",
        optimize_time_shift=True,
        orbital_phase_samples=257,
    ),
    rotation=RotationSpec(
        kind="z_axis",
        optimize_angle=True,
        angle_bounds=(-np.pi, np.pi),
    ),
)

raw_result = fixed_candidate_fitting_factor(
    target_modes,
    candidate_modes,
    config=comparison,
)
raw_rms = normalized_rms_residue(
    target_modes,
    candidate_modes,
    selected_modes,
    raw_result,
)
print_fitting_result("Without BMS-frame preprocessing", raw_result, raw_rms)


## Diagnostics-only BMS-frame preprocessing

In [ ]:
target_checked, candidate_checked, frame_report = preprocess_bms_frame(
    target_modes,
    candidate_modes,
    diagnostics=BMS_DIAGNOSTICS,
)


diagnostic_result = fixed_candidate_fitting_factor(
    target_checked,
    candidate_checked,
    config=comparison,
)
diagnostic_rms = normalized_rms_residue(
    target_checked,
    candidate_checked,
    selected_modes,
    diagnostic_result,
)

print("Target energy radiated:", frame_report.target_diagnostics.energy_radiated)
print("Candidate energy radiated:", frame_report.candidate_diagnostics.energy_radiated)
print("Target radiated linear momentum:", frame_report.target_diagnostics.radiated_linear_momentum)
rotation_details = frame_report.transforms.get("rotation_details", {})
if rotation_details:
    print("Candidate radiated linear momentum before sync:", rotation_details["candidate_vector_before"])
    print("Candidate radiated linear momentum after sync:", rotation_details["candidate_vector_after"])
else:
    print("Candidate radiated linear momentum:", frame_report.candidate_diagnostics.radiated_linear_momentum)
print("BMS-frame preprocessing transform applied:", frame_report.transforms["applied"])
print("Applied operations:", frame_report.transforms["applied_operations"])
print("BMS-frame compatibility:", frame_report.compatibility["compatible"])
print("Failed diagnostic checks:", frame_report.compatibility["failed_checks"])
charge_report = frame_report.charge_diagnostics
print("Low-order charge diagnostics available:", charge_report["available"])
if charge_report["available"]:
    charge_comparison = charge_report["comparison"]
    print("Energy proxy comparison:", charge_comparison["supermomentum_l0_energy"])
    print("Linear momentum proxy comparison:", charge_comparison["supermomentum_l1_linear_momentum"])
    print("Angular momentum proxy comparison:", charge_comparison["angular_momentum"])
    print("Absolute charges computed:", frame_report.target_diagnostics.charge_diagnostics["absolute_bms_charges_computed"])
print(frame_report.assumptions["pn_eob_phenom_frame_comment"])
print()
print_fitting_result("With diagnostics-only BMS preprocessing", diagnostic_result, diagnostic_rms)


## Optional approximate BMS-frame synchronization

In [ ]:
RUN_APPROXIMATE_BMS_SYNC = False
BMS_SYNC_ROTATION = "align_radiated_angular_momentum"

sync_result = None
sync_rms = None
sync_report = None

if RUN_APPROXIMATE_BMS_SYNC:
    print(f"Applying opt-in approximate BMS-frame sync: {BMS_SYNC_ROTATION}")
    print("This is not a guaranteed full BMS transformation and can degrade mode matches.")
    target_sync, candidate_sync, sync_report = preprocess_bms_frame(
        target_modes,
        candidate_modes,
        diagnostics=BMS_DIAGNOSTICS,
        rotation=BMS_SYNC_ROTATION,
    )
    sync_result = fixed_candidate_fitting_factor(
        target_sync,
        candidate_sync,
        config=comparison,
    )
    sync_rms = normalized_rms_residue(
        target_sync,
        candidate_sync,
        selected_modes,
        sync_result,
    )
    print("Applied operations:", sync_report.transforms["applied_operations"])
    print("Rotation details:", sync_report.transforms.get("rotation_details"))
    print_fitting_result("With approximate BMS-frame synchronization", sync_result, sync_rms)
    wrapped_result = bms_synchronized_fixed_candidate_fitting_factor(
        target_modes,
        candidate_modes,
        preprocessing={"diagnostics": BMS_DIAGNOSTICS, "rotation": BMS_SYNC_ROTATION},
        comparison=comparison,
    )
    print("Wrapper match:", wrapped_result.match)
    print("Wrapper phase alignment:", wrapped_result.diagnostics["bms_frame_synchronization"]["phase_alignment_after_preprocessing"])
else:
    print("Approximate BMS-frame synchronization skipped. Set RUN_APPROXIMATE_BMS_SYNC = True to run it.")


## Direct comparison

In [ ]:
print("Diagnostics-only delta match:", diagnostic_result.match - raw_result.match)
print("Diagnostics-only delta mismatch:", diagnostic_result.mismatch - raw_result.mismatch)
print("Diagnostics-only delta RMS residue:", diagnostic_rms - raw_rms)
if sync_result is not None:
    print("Sync delta match:", sync_result.match - raw_result.match)
    print("Sync delta mismatch:", sync_result.mismatch - raw_result.mismatch)
    print("Sync delta RMS residue:", sync_rms - raw_rms)

labels = ["raw", "diagnostics-only"]
matches = [raw_result.match, diagnostic_result.match]
mismatches = [raw_result.mismatch, diagnostic_result.mismatch]
rms_values = [raw_rms, diagnostic_rms]
if sync_result is not None:
    labels.append("BMS-sync")
    matches.append(sync_result.match)
    mismatches.append(sync_result.mismatch)
    rms_values.append(sync_rms)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].bar(labels, matches)
axes[0].set_ylabel("match")
axes[1].bar(labels, mismatches)
axes[1].set_ylabel("mismatch")
axes[2].bar(labels, rms_values)
axes[2].set_ylabel("normalized RMS residue")
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
